# 01 · Data Profiling & Quality

This notebook profiles the **pre-cleaned** benchmark dataset before analysis.
Cleaning (outlier removal + unit conversion) was already performed by
`notebooks/01_data_cleaning.ipynb`, which produced `results/results_clean_runs.csv`.

**Cleaning pipeline (upstream):**
- Outliers removed per **(language × benchmark)** group using the 1.5×IQR boxplot fence,
  applied to both CPU energy and execution time
- Units converted: µJ → J, µs → ms, µg → g, Bytes → MB, mW → W

**This notebook covers:**
- Column overview and descriptive statistics
- Missing value check
- Metric distributions
- Data coverage (languages × benchmarks)
- Per-language summary table

**Dataset:** 18 languages × 8 benchmarks, measured with the Green Metrics Tool (GMT).
**Priority metrics:** CPU Energy (J), Memory Energy (J), Execution Time (ms)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import plot_style as ps
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
PARADIGM        = ps.PARADIGM
PARADIGM_COLORS = ps.PARADIGM_COLORS
PARADIGM_ORDER  = ps.PARADIGM_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Column Overview

Extended `describe()` supplemented with median and IQR for the three priority columns.
Median is preferred over mean for right-skewed benchmark distributions.
All values are in human-readable units (J for energy, ms for time).

In [ ]:
priority = [COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME]
labels   = {COL_CPU_ENERGY: 'CPU Energy (J)',
            COL_MEM_ENERGY: 'Mem Energy (J)',
            COL_TIME:       'Time (ms)'}

stats_tbl = df[priority].describe().T
stats_tbl['median'] = df[priority].median()
stats_tbl['IQR']    = df[priority].quantile(0.75) - df[priority].quantile(0.25)
stats_tbl['skew']   = df[priority].skew()
stats_tbl.index     = [labels[c] for c in stats_tbl.index]
stats_tbl.round(4)

## 2. Missing Values

Completeness check across all 16 columns. Benchmark datasets often contain zeros rather
than NaN for metrics that were not triggered (e.g. network bytes on a CPU-only task).

In [ ]:
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("✓ No missing values — dataset is complete.")
else:
    print("Missing values detected:")
    print(null_counts[null_counts > 0])

# Check for zero-only columns (may indicate inactive metrics)
zero_frac = (df[priority] == 0).mean()
print("\nFraction of zeros in priority columns:")
for col, frac in zip(['CPU Energy', 'Mem Energy', 'Time'], zero_frac):
    print(f"  {col}: {frac:.1%}")

## 3. Distributions

Histograms on a log₁₀ scale for the three priority metrics.
Benchmark data is typically right-skewed (a few languages/benchmarks dominate the upper tail).
Values are in J (energy) and ms (time).
A divergence of >20% between mean and median is flagged.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_info = [
    ('CPU Energy', COL_CPU_ENERGY, 'J'),
    ('Memory Energy', COL_MEM_ENERGY, 'J'),
    ('Execution Time', COL_TIME, 'ms'),
]
for ax, (label, col, unit) in zip(axes, metric_info):
    vals = df[col][df[col] > 0]
    log_vals = np.log10(vals)
    ax.hist(log_vals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    mean_v, med_v = vals.mean(), vals.median()
    ax.axvline(np.log10(mean_v), color='red',    linestyle='--', label=f'mean={mean_v:.2f} {unit}')
    ax.axvline(np.log10(med_v),  color='orange', linestyle='-',  label=f'median={med_v:.2f} {unit}')
    ax.set_title(f'{label}')
    ax.set_xlabel(f'log₁₀({unit})')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    if abs(mean_v - med_v) / med_v > 0.20:
        ax.set_title(ax.get_title() + '\n⚠ mean/median diverge >20%')

fig.suptitle('Priority Metric Distributions (log₁₀ scale)', fontsize=13, y=1.02)
plt.tight_layout()
ps.save_fig(fig, '01_distributions')
plt.show()

> **Takeaway:** all three priority metrics are strongly right-skewed (mean ≫ median) — this is why rankings use means while the significance tests stay non-parametric.

## 4. Data Coverage

Heatmap showing the number of runs for each language × benchmark pair.
A uniform count across all cells indicates balanced coverage.

In [ ]:
coverage = df.groupby(['language', 'benchmark']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(coverage, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.4, linecolor='#ccc',
            cbar_kws={'label': 'Number of runs'})
ax.set_title('Data Coverage: Runs per Language × Benchmark', fontsize=13)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Language')
plt.tight_layout()
ps.save_fig(fig, '01_coverage_heatmap')
plt.show()
print(f"Min runs in any cell: {coverage.values.min()}")
print(f"Max runs in any cell: {coverage.values.max()}")
print(f"Mean runs per cell:   {coverage.values.mean():.1f}")

> **Takeaway:** every (language × benchmark) cell keeps 5–10 clean runs after outlier removal — coverage is complete and roughly balanced.

## 5. Summary Table

Per-language descriptive statistics for all three priority metrics in human-readable units.
Mean and median are both reported; large divergence signals skewness within a language.

In [ ]:
summary_tbl = df.groupby('language').agg(
    paradigm      = ('paradigm', 'first'),
    runs          = ('run_id', 'count'),
    cpu_mean_J    = (COL_CPU_ENERGY, 'mean'),
    cpu_median_J  = (COL_CPU_ENERGY, 'median'),
    mem_mean_J    = (COL_MEM_ENERGY, 'mean'),
    mem_median_J  = (COL_MEM_ENERGY, 'median'),
    time_mean_ms   = (COL_TIME, 'mean'),
    time_median_ms = (COL_TIME, 'median'),
).round(4)

# Flag skew
for label, mc, mdc in [('CPU energy', 'cpu_mean_J', 'cpu_median_J'),
                        ('Mem energy', 'mem_mean_J', 'mem_median_J'),
                        ('Time',       'time_mean_ms','time_median_ms')]:
    skew_mask = (abs(summary_tbl[mc] - summary_tbl[mdc]) / summary_tbl[mdc]) > 0.20
    if skew_mask.any():
        print(f"⚠ {label} mean/median diverge >20% for: {list(summary_tbl.index[skew_mask])}")

summary_tbl.sort_values('cpu_median_J')